In [1]:
!pip install PyMuPDF numpy opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 40.2 MB/s eta 0:00:00


# 1. The mental model

`header_footer_v3.py` does `detection + cropping at the PDF level`. The output is a PDF with the original pages but with CropBoxes set to the kept region. It’s lossless and lightweight (~same file size as input).

`page_export_v2.py` does `rasterization at high resolution`. It takes a stripped PDF and produces OCR-grade images. Two operations: full pages (for the body), and footers (the discarded margins, kept for reference or separate OCR).

Run detection once, raster from its output many times if needed.

## Recommended single-book workflow

In [9]:
from pathlib import Path
from header_footer_v3 import strip_pdf, Params
from page_export_v2 import export_pages_as_images, extract_footers_pdf

src = "Memoirs.pdf"
stem = Path(src).stem
out = Path("out")

# Create the output directory if it doesn't exist
out.mkdir(parents=True, exist_ok=True)

# 1. Detect & strip — produces a CropBox-cropped PDF.
print("Stripping header/footer from:", src, ".....")
strip_pdf(src, out / f"{stem}_stripped.pdf")
print(f"✅ {stem}_stripped.pdf completed!\n")

# 2. Render the kept body pages at 400 DPI for the OCR engine.
print("Exporting pages as images from:", src, ".....")
export_pages_as_images(
    out / f"{stem}_stripped.pdf",
    out / f"{stem}_pages",
    dpi=400,
    zip_path=out / f"{stem}_pages.zip",
)
print(f"🎆 {stem}_pages.zip completed!\n")

# 3. Collect the discarded footers — both as a bound PDF (for human review)
#    and as standalone images (for separate OCR if needed).
print("Exporting footers from:", src, ".....")
extract_footers_pdf(
    src,                                       # ORIGINAL, not stripped
    out / f"{stem}_footers.pdf",
    img_dir=out / f"{stem}_footers_imgs",
    images_dpi=400,
    zip_path=out / f"{stem}_footers_imgs.zip",
)
print(f"🧾 {stem}_footers_imgs.zip completed!\n")


Stripping header/footer from: Memoirs.pdf .....
✅ Memoirs_stripped.pdf completed!

Exporting pages as images from: Memoirs.pdf .....
🎆 Memoirs_pages.zip completed!

Exporting footers from: Memoirs.pdf .....
🧾 Memoirs_footers_imgs.zip completed!



In [10]:
from pathlib import Path
import os

out_dir = Path("out")
total_size = 0

print(f"Files in '{out_dir}' and their sizes:")
for file_path in out_dir.rglob("*"):
    if file_path.is_file():
        size_bytes = os.path.getsize(file_path)
        size_mb = size_bytes / (1024 * 1024)
        print(f"- {file_path.name}: {size_mb:.2f} MB")
        total_size += size_bytes

total_size_mb = total_size / (1024 * 1024)
print(f"\nTotal size of all files in '{out_dir}': {total_size_mb:.2f} MB")

Files in 'out' and their sizes:
- Memoirs_footers_imgs.zip: 26.99 MB
- Memoirs_stripped.pdf: 29.17 MB
- Memoirs_footers.pdf: 22.17 MB
- Memoirs_pages.zip: 1016.03 MB
- footer_p029.png: 0.22 MB
- footer_p114.png: 0.48 MB
- footer_p135.png: 0.27 MB
- footer_p186.png: 0.13 MB
- footer_p179.png: 0.07 MB
- footer_p115.png: 0.64 MB
- footer_p193.png: 0.37 MB
- footer_p106.png: 0.95 MB
- footer_p103.png: 0.37 MB
- footer_p118.png: 0.21 MB
- footer_p100.png: 1.05 MB
- footer_p233.png: 0.06 MB
- footer_p049.png: 0.54 MB
- footer_p189.png: 0.07 MB
- footer_p022.png: 0.02 MB
- footer_p221.png: 0.07 MB
- footer_p004.png: 0.04 MB
- footer_p099.png: 0.50 MB
- footer_p174.png: 0.06 MB
- footer_p028.png: 0.12 MB
- footer_p138.png: 0.06 MB
- footer_p242.png: 0.05 MB
- footer_p111.png: 0.62 MB
- footer_p044.png: 0.41 MB
- footer_p157.png: 0.15 MB
- footer_p234.png: 0.05 MB
- footer_p190.png: 0.78 MB
- footer_p130.png: 0.36 MB
- footer_p023.png: 0.05 MB
- footer_p163.png: 0.04 MB
- footer_p019.png: 0.25 

In [12]:
import zipfile
import os
import shutil

# Define paths
original_zip_path = Path("out/Memoirs_pages.zip")
output_split_dir = Path("out/Memoirs_pages_split")
temp_extract_dir = Path("out/temp_extracted_pages")

# Create necessary directories
output_split_dir.mkdir(parents=True, exist_ok=True)
temp_extract_dir.mkdir(parents=True, exist_ok=True)

print(f"Extracting {original_zip_path} to {temp_extract_dir}...")
with zipfile.ZipFile(original_zip_path, 'r') as zip_ref:
    zip_ref.extractall(temp_extract_dir)
print("Extraction complete.")

# Get a list of all extracted files
all_files = sorted([f for f in temp_extract_dir.iterdir() if f.is_file()])

# Define target size for each new zip file (e.g., 50 MB)
# This is an approximate target, actual sizes might vary based on file compression
target_chunk_size_mb = 250
target_chunk_size_bytes = target_chunk_size_mb * 1024 * 1024

current_chunk_files = []
current_chunk_size = 0
zip_part_index = 0

print(f"Splitting into approximately {target_chunk_size_mb} MB chunks...")
for file_path in all_files:
    file_size = os.path.getsize(file_path)

    if current_chunk_size + file_size > target_chunk_size_bytes and current_chunk_files:
        # If adding this file exceeds target and we already have files in chunk, zip current chunk
        zip_part_index += 1
        new_zip_name = output_split_dir / f"Memoirs_pages_part_{zip_part_index:03d}.zip"
        with zipfile.ZipFile(new_zip_name, 'w', zipfile.ZIP_DEFLATED) as new_zip:
            for f in current_chunk_files:
                new_zip.write(f, f.name) # Write with just the filename, not full path
        print(f"Created {new_zip_name} ({current_chunk_size / (1024 * 1024):.2f} MB)")

        # Reset for next chunk
        current_chunk_files = [file_path]
        current_chunk_size = file_size
    else:
        # Add file to current chunk
        current_chunk_files.append(file_path)
        current_chunk_size += file_size

# Zip any remaining files in the last chunk
if current_chunk_files:
    zip_part_index += 1
    new_zip_name = output_split_dir / f"Memoirs_pages_part_{zip_part_index:03d}.zip"
    with zipfile.ZipFile(new_zip_name, 'w', zipfile.ZIP_DEFLATED) as new_zip:
        for f in current_chunk_files:
            new_zip.write(f, f.name)
    print(f"Created {new_zip_name} ({current_chunk_size / (1024 * 1024):.2f} MB)")

# Clean up temporary extracted directory
print(f"Cleaning up temporary directory {temp_extract_dir}...")
shutil.rmtree(temp_extract_dir)
print("Splitting complete. New zip files are in 'out/Memoirs_pages_split/'.")

Extracting out/Memoirs_pages.zip to out/temp_extracted_pages...
Extraction complete.
Splitting into approximately 250 MB chunks...
Created out/Memoirs_pages_split/Memoirs_pages_part_001.zip (248.70 MB)
Created out/Memoirs_pages_split/Memoirs_pages_part_002.zip (248.38 MB)
Created out/Memoirs_pages_split/Memoirs_pages_part_003.zip (245.00 MB)
Created out/Memoirs_pages_split/Memoirs_pages_part_004.zip (234.89 MB)
Created out/Memoirs_pages_split/Memoirs_pages_part_005.zip (57.99 MB)
Cleaning up temporary directory out/temp_extracted_pages...
Splitting complete. New zip files are in 'out/Memoirs_pages_split/'.


That’s it. Three calls give you everything: stripped PDF, 400 DPI page images, footer PDF, footer images. The directories are optional intermediates — if you only want the zips, delete the dirs after.

The crucial subtlety: extract_footers_pdf takes the ORIGINAL PDF, not the stripped one. The stripped PDF has the footers cropped out — there’s nothing to extract from it. The detector re-runs on the original to locate the same footer regions it discarded.

## Tuning Params (when defaults don’t fit)

90% of books work with Params(). The other 10% break in predictable ways. Here’s a triage table.

|Symptom                                                                         |Knob                                                                               |Direction                               |
|------------------------------------------------------------------ |------------------------------------------------------------------|-----------------------------------|
|Footer rule missed → footer leaks into body                  |`rule_thickness_max_px`                                                |raise (12 → 16 → 20)          |
|Real text passes as a “rule” → page over-cropped          |`rule_isolation_px`                                                          |raise (18 → 25)                    |
|Or                                                                                     |`rule_isolation_max_ink_frac`                                        |lower (0.02 → 0.01)            |
|Header detected but page number kept                            |`narrow_width_ratio`                                                      |lower (0.65 → 0.5)              |
|Body text classified as header                                          |`narrow_width_ratio`                                                      |raise (0.65 → 0.8)                |
|First/last page over-cropped (chapter openers, blank ends)|leave alone — sparse-page guard handles this; if not, raise `min_text_height_frac`|                  

<p align="center">
  <img src="Table.jpg" width="400">
</p>


When debugging a single PDF, run detect_margins(page, p, verbose=True) and read m.notes. The notes are populated only when verbose=True (zero overhead in production), and they tell you which signals fired and which were rejected.

# 2. Batch processing

## For a directory of books:

In [13]:
from pathlib import Path
souce_bk = Path("source_books")

# Create the output directory if it doesn't exist
souce_bk.mkdir(parents=True, exist_ok=True)

In [14]:
from pathlib import Path
from header_footer_v3 import strip_pdf
from page_export_v2 import export_pages_as_images, extract_footers_pdf

def process_one(src: Path, out_root: Path) -> None:
    stem = src.stem
    out = out_root / stem
    out.mkdir(parents=True, exist_ok=True)
    stripped = out / f"{stem}_stripped.pdf"
    strip_pdf(src, stripped)
    export_pages_as_images(stripped, out / "pages", dpi=400,
                           zip_path=out / f"{stem}_pages.zip")
    extract_footers_pdf(src, out / f"{stem}_footers.pdf",
                        img_dir=out / "footers", images_dpi=400,
                        zip_path=out / f"{stem}_footers.zip")

for pdf in Path("source_books").glob("*.pdf"):
    try:
        process_one(pdf, Path("processed"))
    except Exception as e:
        print(f"FAILED {pdf.name}: {e}")


Wrap each book in a try/except — one bad PDF shouldn’t kill the run. For parallel processing, use concurrent.futures.ProcessPoolExecutor (CPU-bound, GIL-bound; threads won’t help). 4-8 workers is usually the sweet spot.

# Format choices

For OCR engines: stick with PNG (the default). Tesseract, Kraken, and EasyOCR all read PNG natively and lossless preservation matters for sub-pixel diacritics in Arabic. JPEG quality artifacts compound across pages.

For storage: TIFF if your downstream pipeline expects it (some scientific OCR tools do). Same lossless quality, slightly larger files than PNG.

⚠️ Avoid: JPEG for OCR. The exporter supports it for completeness but quality 95 still leaves visible ringing around fine strokes.

# What to skip when

* ⚠️ Don’t re-export images you already have. The stripped PDF is the source of truth; rasterization is reproducible. Keep the stripped PDF in version control or backup, regenerate images on demand.
* ⚠️ Don’t run footer extraction if you don’t need footers. It re-renders every page at p.dpi and then again at images_dpi for each detected footer. On a 300-page book that’s noticeable.
* ⚠️ Don’t increase DPI past 400 unless your OCR engine specifically benefits. 400 DPI on A4 is already 3300×4660 px per page. 600 DPI doubles file size for marginal OCR gains on standard-print Arabic books.

# When v3 is wrong

If the detector misclassifies a page, your fastest path is:
1. Run detect_margins(page, Params(), verbose=True) and read the notes.
1. Render the page binarized with detected runs/rules overlaid (the diagnostic pattern from the v2→v3 debugging session).
1. Decide if it’s a parameter-tuning case (use the table above) or a structural detector limitation (decorated chapter pages, multi-column layouts, very tight margins).

For one-off problem pages, the lossless cropbox mode is your friend — if v3 gets it 95% right, you can manually fix the remaining pages in any PDF editor by adjusting the CropBox before exporting images. Faster than tuning Params for an outlier.

# 3. A small caveat

`page_export_v2.py` doesn’t validate that you’ve run `strip_pdf` first. If you point it at the original PDF directly, it will happily render every full page at 400 DPI — no crops, no error. The CropBox just defaults to the MediaBox. Useful occasionally (full-page exports for archival), but worth being aware of.​​​

In [8]:
from pathlib import Path
from header_footer_v3 import strip_pdf, Params
from page_export_v2 import export_pages_as_images, extract_footers_pdf

src = "Chapter 1.pdf"
stem = Path(src).stem
out = Path("out_full")

# Create the output directory if it doesn't exist
out.mkdir(parents=True, exist_ok=True)

# Render the kept body pages at 400 DPI for the OCR engine.
print("Exporting source as 400DPI images from:", src, ".....")
export_pages_as_images(
    src,
    out / f"{stem}_pages",
    dpi=400,
    zip_path=out / f"{stem}_unstripped.zip",
)
print(f"🎆 {stem}_pages.zip completed!\n")


Exporting source as 400DPI images from: Chapter 1.pdf .....
🎆 Chapter 1_pages.zip completed!

